# Week 33, Day 2: Collections and Optional

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w033d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
ages = {"Ada": 31, "Bo": 25}
for name in ["Ada", "Cy"]:
    age = ages.get(name)
    print(name, age, age is None)

In [ ]:
def word_counts(text: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    for word in text.lower().split():
        counts[word] = counts.get(word, 0) + 1
    return counts

def pairs(names: list[str], ages: list[int]) -> list[tuple[str, int]]:
    return list(zip(names, ages))

def lowest(*marks: int) -> tuple[int, ...]:
    return tuple(sorted(marks)[:3])

print(word_counts("the cat and the hat"))
print(pairs(["Ada", "Bo"], [31, 25]), lowest(70, 55, 90, 62))

In [ ]:
def find_price(prices: dict[str, int], item: str) -> int | None:
    return prices.get(item)

def label(name: str | None = None) -> str:
    if name is None:
        return "Guest"
    return name.title()

prices = {"rice": 450, "oil": 350}
print(find_price(prices, "rice"), find_price(prices, "salt"), label(), label("ada"))

In [ ]:
def as_pence(amount: int | str) -> int:
    if isinstance(amount, int):
        return amount                         # here, amount is an int
    text = amount.strip().lstrip("£")         # here, amount is a str
    pounds, _, pence = text.partition(".")
    return int(pounds) * 100 + int(pence.ljust(2, "0") or 0)

print(as_pence(1250), as_pence("£12.50"), as_pence(" 3.5 "), as_pence("7"))

In [ ]:
from collections.abc import Iterable, Mapping, Sequence

def total(prices: Iterable[int]) -> int:
    return sum(prices)

def middle(values: Sequence[float]) -> float:
    ordered = sorted(values)
    n = len(ordered)
    return ordered[n // 2] if n % 2 else (ordered[n // 2 - 1] + ordered[n // 2]) / 2

def cheapest(prices: Mapping[str, int]) -> list[str]:
    low = min(prices.values())
    return sorted(name for name, p in prices.items() if p == low)

print(total([1, 2]), total((3, 4)), total(range(5)), total(p * 2 for p in [5, 5]))
print(middle([3, 1, 2]), middle((4, 1, 3, 2)), cheapest({"rice": 450, "salt": 90, "pepper": 90}))

In [ ]:
Cell = tuple[int, int]
Grid = list[list[str]]

def neighbours(cell: Cell, grid: Grid) -> list[Cell]:
    r, c = cell
    found = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        if 0 <= r + dr < len(grid) and 0 <= c + dc < len(grid[0]):
            found.append((r + dr, c + dc))
    return found

grid: Grid = [list("..#"), list("...")]
print(neighbours((0, 0), grid), neighbours((1, 1), grid))

> **Key idea.** Collection hints name the contents: `list[str]`, `set[int]`, `dict[str, int]`, `tuple[str, int]` for a fixed shape and `tuple[int, ...]` for any length; empty collections need a hint. `X | None` marks a value that might be missing; narrow it with `if x is None:` (or a default, or raising) before using it, and the checker follows along. `int | str` is a union; narrow it with `isinstance`. Accept `Iterable`, `Sequence` or `Mapping` in parameters, return precise types. Name long types with aliases (`Grid = list[list[str]]`); `Any` switches checking off, `object` doesn't.

### Exercise: Hints for collections

Add hints to the three functions without changing what they do: `word_lengths` takes text and returns a dictionary from each word to its length; `tagged` takes a list of names and a tag and returns a list of (name, tag) pairs; `initials` takes a list of names and returns the set of their first letters.

In [ ]:
def word_lengths(text):
    return {word: len(word) for word in text.split()}

def tagged(names, tag):
    return [(name, tag) for name in names]

def initials(names):
    return {name[0] for name in names if name}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"the hints\"\n    from typing import get_type_hints as h\n    assert h(word_lengths) == {\"text\": str, \"return\": dict[str, int]}, f\"word_lengths: {h(word_lengths)}\"\n    assert h(tagged) == {\"names\": list[str], \"tag\": str, \"return\": list[tuple[str, str]]}, f\"tagged: {h(tagged)}\"\n    assert h(initials) == {\"names\": list[str], \"return\": set[str]}, f\"initials: {h(initials)}\"\n    assert word_lengths(\"a bb\") == {\"a\": 1, \"bb\": 2} and tagged([\"x\"], \"t\") == [(\"x\", \"t\")] and initials([\"Ada\", \"Bo\", \"Ali\"]) == {\"A\", \"B\"}")

### Exercise: Maybe a price

Write `find_price(prices: dict[str, int], item: str) -> int | None`, which gives the price or `None`, and `known_total(prices: dict[str, int], items: list[str]) -> tuple[int, list[str]]`, which adds up the prices of the items it knows (using `find_price`) and also returns, in order, the items it didn't know.

In [ ]:
def find_price(prices, item):
    ___

def known_total(prices, items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lookup():\n    \"prices\"\n    from typing import get_type_hints as h\n    assert h(find_price) == {\"prices\": dict[str, int], \"item\": str, \"return\": int | None}, f\"find_price: {h(find_price)}\"\n    assert h(known_total) == {\"prices\": dict[str, int], \"items\": list[str], \"return\": tuple[int, list[str]]}, f\"known_total: {h(known_total)}\"\n    p = {\"rice\": 450, \"oil\": 350, \"salt\": 0}\n    assert find_price(p, \"oil\") == 350 and find_price(p, \"yam\") is None and find_price(p, \"salt\") == 0\n    assert known_total(p, [\"rice\", \"yam\", \"salt\", \"rice\", \"egg\"]) == (900, [\"yam\", \"egg\"]) and known_total(p, []) == (0, [])")

### Exercise: Scores that might not be there

Write two functions with these hints (import `Iterable` from `collections.abc`):

- `parse_score(text: str) -> int | None`: the whole number in the text (spaces at the ends ignored) if it's from 0 to 100, otherwise `None` (for blanks, words like `"absent"`, numbers out of range, and decimals);
- `average(lines: Iterable[str]) -> float | None`: the mean of the valid scores, rounded to 1 place, ignoring invalid lines; `None` if there are no valid scores. It must work for any iterable, including a generator, so loop over it only once.

In [ ]:
from collections.abc import Iterable

def parse_score(text):
    ___

def average(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scores():\n    \"scores\"\n    from typing import get_type_hints as h\n    from collections.abc import Iterable as It\n    assert h(parse_score) == {\"text\": str, \"return\": int | None} and h(average) == {\"lines\": It[str], \"return\": float | None}\n    assert [parse_score(t) for t in [\"72\", \" 0 \", \"100\", \"101\", \"-3\", \"absent\", \"\", \"7.5\"]] == [72, 0, 100, None, None, None, None, None]\n    assert average([\"70\", \"absent\", \"81\", \"\"]) == 75.5 and average([\"x\", \"200\"]) is None and average([]) is None\n    assert average(line for line in [\"50\", \"60\"]) == 55.0 and average((\"0\",)) == 0.0")

### Exercise: Money in several forms

Write `to_pence(amount: int | float | str) -> int`. An `int` is already pence. A `float` is pounds (`12.5` is 1250; round to the nearest penny). A `str` may be pounds with a `£` sign and up to two decimal places (`"£12.50"`, `"£3"`, `"£0.5"`) or pence with a `p` at the end (`"75p"`); spaces at the ends are ignored, and anything else raises `ValueError`. Any other type (including `bool`) raises `TypeError`.

In [ ]:
import re

def to_pence(amount):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pence():\n    \"conversions\"\n    from typing import get_type_hints as h\n    assert h(to_pence) == {\"amount\": int | float | str, \"return\": int}, f\"hints: {h(to_pence)}\"\n    assert [to_pence(x) for x in [1250, 12.5, 0.1 + 0.2, \"\u00a312.50\", \" \u00a33 \", \"\u00a30.5\", \"75p\", \"0p\"]] == [1250, 1250, 30, 1250, 300, 50, 75, 0]\n    for bad in [\"12.50\", \"\u00a31.234\", \"\u00a3\", \"p\", \"\u00a3-3\", \"seven\"]:\n        try:\n            to_pence(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass\n    for bad in [True, None, [1]]:\n        try:\n            to_pence(bad)\n            assert False, f\"{bad!r} should raise TypeError\"\n        except TypeError:\n            pass")

### Bug hunt: Shouting at nobody

mypy reports `error: Item "None" of "str | None" has no attribute "upper"  [union-attr]`, and the program crashes with `AttributeError: 'NoneType' object has no attribute 'upper'` for the guest. A missing name should become `"GUEST"`. Fix `badge` so it prints `ADA` then `GUEST`.

In [ ]:
def badge(name: str | None) -> str:
    return name.upper()

for name in ["Ada", None]:
    print(badge(name))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"badges\"\n    expect_output(\"ADA\\nGUEST\")\n    assert badge(\"\") == \"\", \"an empty name isn't a missing one\"")

### Bug hunt: Falling off the end

mypy reports `error: Missing return statement  [return]` for `first_word`, and the program crashes with `AttributeError: 'NoneType' object has no attribute 'title'` on the blank line. A blank line's first word should be the empty string. Fix `first_word` so it prints `Hello`, an empty line, then `Bye`.

In [ ]:
def first_word(text: str) -> str:
    if text.strip():
        return text.split()[0]

for line in ["hello world", "   ", "bye now"]:
    print(first_word(line).title())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"words\"\n    expect_output(\"Hello\\n\\nBye\")")

### Bug hunt: Counting from nothing

mypy reports `Unsupported operand types for + ("None" and "int")` for the line that counts visits, and the program crashes the first time a page is visited. Fix that line, keeping the dictionary, so it prints `{'home': 2, 'shop': 1}`.

In [ ]:
def count_visits(pages: list[str]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for page in pages:
        counts[page] = counts.get(page) + 1
    return counts

print(count_visits(["home", "shop", "home"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"counted\"\n    expect_output(\"{'home': 2, 'shop': 1}\")")

### Puzzle: A safe lookup

Arrange the lines into a function that gives a member's town, or `"unknown"` if the member isn't listed. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from collections.abc import Iterable

def f(xs: Iterable[int]) -> list[int]:
    return sorted(xs)

def g(d: dict[str, int], k: str) -> int | None:
    return d.get(k)

Pair = tuple[str, int]

p = ___   # f({3, 1, 2})
q = ___   # g({"a": 0}, "a")
r = ___   # g({"a": 0}, "b")
s = ___   # Pair == tuple[str, int]
t = ___   # f(range(3, 0, -1))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == ([1, 2, 3], 0, None), \"p to r\"\n    assert s is True and t == [1, 2, 3]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a typed mark book

Lines of a mark book look like `Ada: 70, 81, 64` (a name, a colon, then marks separated by commas; a student may have no marks yet: `Bo:`). Write these fully annotated functions (use `Iterable` and `Mapping` from `collections.abc`):

- `parse_line(line: str) -> tuple[str, list[int]] | None`: the name (spaces stripped) and marks, or `None` for a blank line. A line without a colon, an empty name, or a mark that isn't a whole number from 0 to 100 raises `ValueError`.
- `load(lines: Iterable[str]) -> dict[str, list[int]]`: every student's marks, skipping blank lines; a name that appears twice has its marks joined.
- `mean(marks: list[int]) -> float | None`: rounded to 1 place, or `None` for no marks.
- `top(book: Mapping[str, list[int]]) -> tuple[str, float] | None`: the student with the highest mean (the first alphabetically on a tie), with that mean; students without marks are ignored; `None` if nobody has marks.
- `report(book: Mapping[str, list[int]]) -> list[str]`: one line per student, alphabetically, like `Ada: 71.7` or `Bo: no marks`.

In [ ]:
from collections.abc import Iterable, Mapping

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hints():\n    \"hints\"\n    from typing import get_type_hints as h\n    from collections.abc import Iterable as It, Mapping as Mp\n    assert h(parse_line) == {\"line\": str, \"return\": tuple[str, list[int]] | None}\n    assert h(load) == {\"lines\": It[str], \"return\": dict[str, list[int]]} and h(mean) == {\"marks\": list[int], \"return\": float | None}\n    assert h(top) == {\"book\": Mp[str, list[int]], \"return\": tuple[str, float] | None} and h(report) == {\"book\": Mp[str, list[int]], \"return\": list[str]}\ndef test_behaviour():\n    \"the mark book\"\n    assert parse_line(\" Ada : 70, 81 ,64\") == (\"Ada\", [70, 81, 64]) and parse_line(\"Bo:\") == (\"Bo\", []) and parse_line(\"   \") is None\n    for bad in [\"Ada 70\", \": 50\", \"Ada: 101\", \"Ada: 7.5\", \"Ada: 70,,80\"]:\n        try:\n            parse_line(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass\n    book = load([\"Ada: 70, 81, 64\", \"\", \"Bo:\", \"Chi: 90\", \"Ada: 75\", \"Dami: 80, 85\"])\n    assert book == {\"Ada\": [70, 81, 64, 75], \"Bo\": [], \"Chi\": [90], \"Dami\": [80, 85]}\n    assert mean([70, 81, 64]) == 71.7 and mean([]) is None\n    assert top(book) == (\"Chi\", 90.0) and top({\"Bo\": []}) is None and top({\"Z\": [80], \"A\": [80]}) == (\"A\", 80.0)\n    assert report(book) == [\"Ada: 72.5\", \"Bo: no marks\", \"Chi: 90.0\", \"Dami: 82.5\"]")

## Warm-up

`ages.get(name)` gives an `int` when the name is there and `None` when it isn't. So its type isn't really `int`; it's "an `int` or `None`". Code that forgets the `None` case (say, `age + 1`) works in every test where the name exists, and crashes the first time it doesn't. Today's hints make that possibility visible, and a type checker makes sure you've dealt with it.

## Collections

A collection's hint says what kind of collection it is **and** what it holds, in square brackets:

| Hint | Means |
|---|---|
| `list[str]` | a list of strings |
| `set[int]` | a set of whole numbers |
| `dict[str, int]` | a dictionary with string keys and int values |
| `tuple[str, int]` | a tuple of exactly two items: a string, then an int |
| `tuple[int, ...]` | a tuple of any length, all ints |
| `list[dict[str, float]]` | a list of dictionaries, each from strings to floats |

- Lists, sets and dictionaries normally hold one kind of thing, so their hints have one type (two for a dictionary). Tuples are often records with a fixed shape, so `tuple[str, int]` lists each position.
- `*marks: int` means every extra positional argument is an `int`; inside, `marks` is a `tuple[int, ...]`.
- An **empty** collection is the one place a variable usually needs a hint: from `counts = {}` a checker can't tell what will go in it. mypy says so: `Need type annotation for "names" (hint: "names: list[<type>] = ...")  [var-annotated]`.

Once the checker knows what a collection holds, it checks what goes in and what comes out:

```python
scores: list[int] = []
scores.append("ten")
pair: tuple[str, int] = ("Ada", "31")
```

```text
scores.py:2: error: Argument 1 to "append" of "list" has incompatible type "str"; expected "int"  [arg-type]
scores.py:3: error: Incompatible types in assignment (expression has type "tuple[str, str]", variable has type "tuple[str, int]")  [assignment]
```

In older code you'll see `List[str]`, `Dict[str, int]` and `Tuple[str, int]`, imported from `typing`. They mean the same; since Python 3.9 the built-in names work directly, and are preferred.

## Values that might be missing

`X | None` means "an `X`, or `None`". It's the hint for any function that might not find what it was asked for, and for parameters that are optional in meaning, not just in having a default:

You'll also see `Optional[int]` (from `typing`) in older code; it means exactly `int | None`.

The checker won't let you use a possibly-`None` value as if it were definitely there. Here's what mypy says about two careless functions:

```python
def shout(name: str | None) -> str:
    return name.upper()

ages = {"Ada": 31}
next_year = ages.get("Ada") + 1
```

```text
shout.py:2: error: Item "None" of "str | None" has no attribute "upper"  [union-attr]
shout.py:5: error: Unsupported operand types for + ("None" and "int")  [operator]
shout.py:5: note: Left operand is of type "int | None"
```

The fix is to deal with `None` first. The checker follows your `if` statements and knows that after `if name is None: return ...`, `name` can't be `None` any more. This is called **narrowing**:

```python
def shout(name: str | None) -> str:
    if name is None:
        return ""
    return name.upper()          # here, name is a str
```

Other ways to narrow: `if name is not None:` around the code that uses it; giving a default (`ages.get("Ada", 0)` has type `int`); and raising an error (`if name is None: raise ValueError(...)`). Be careful with `if name:`: it also treats `""`, `0` and empty lists as missing, which may not be what you mean.

A related bug that type checkers catch: a function that's declared to return a `str` but can fall off the end without returning (and so returns `None`). mypy reports it as `Missing return statement  [return]`.

## One of several types

`int | str` means "an `int` or a `str`" (older code: `Union[int, str]`). Narrow it with `isinstance`, which the checker also follows:

Unions are useful at the edges of a program, where data really does come in several forms. Inside, prefer one type: a function that takes `int | str | float | None` everywhere makes every caller handle every case. Turn values into one type as early as you can, as `as_pence` does.

## Accept broadly, return precisely

Should a function that adds up some numbers take a `list[int]`? Then a caller with a tuple, a set, a `range` or a generator can't use it without converting. The `collections.abc` types from last week work as hints too, and describe what the function **needs**:

| Hint | Accepts | Use when the function... |
|---|---|---|
| `Iterable[T]` | anything you can loop over | only loops over it, once |
| `Sequence[T]` | lists, tuples, strings, ranges | needs `len`, indexing or more than one pass |
| `Mapping[K, V]` | dictionaries and dictionary-like objects | only reads keys and values |
| `MutableMapping[K, V]` | dictionaries | also changes them |

The rule of thumb: **parameters as general as the function can handle; return types as specific as you can promise.** `cheapest` accepts any mapping but promises a `list[str]`, so callers can index it, append to it and sort it.

## Aliases and Any

When a type gets long and is used in several places, give it a name. A **type alias** is just an assignment:

(Python 3.12 added a dedicated form, `type Grid = list[list[str]]`, which you'll see in newer code.)

Finally, `typing.Any` means "any type at all, and don't check it". A value of type `Any` can be used in any way, and the checker stays silent, so it switches checking off for that value. It's the honest hint for data whose shape you really don't know yet (JSON straight from `json.loads`), and a sign of unfinished work elsewhere. If you mean "anything, but I'll only use it in ways every object supports", the hint is `object`, which the checker *does* check.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at a function you've written that returns `None` when something isn't found, and give it an honest `X | None` return hint. Then check every place that calls it: does each one deal with `None`? Tomorrow: running mypy on your own computer, reading its reports, and setting how strict it should be.